# 演習2. スレッドセーフなキューを使う

演習1で、パイプラインは「段」を並べて**ずらして重ねる**形だと分かりました。
残っているのは、**段と段を何でつなぐか**です。

```
[ Read ] --?--> [ Infer ] --?--> [ Show ]
```

この `?` が **キュー（待ち行列）** です。ただし、**ふつうの `std::queue` は使えません。**
まずそれを確かめ、そのあと**この演習で使うキュー**を渡します。

> **この演習でやることは「キューを使えるようになる」だけ**です。キューの中身は読まなくて構いません。

## 2-1. 【予測クイズ】守らずに共有すると、どうなるか

2つのスレッドが、守られていない共有データを触ります。

- **①** 共有カウンタを、2人が 100万回ずつ増やす（期待値 200万）
- **②** 共有の「次に取る番号」を、2人が奪い合って進める

②の中身はこれだけです。**1行ずつは正しく見えます。**

```cpp
if (idx >= NMAX) return;   // ← まだ残っているか確かめて…
int i = idx;               // ← 自分の分として取って…
idx++;                     // ← 番号を1つ進める
```

**実行前に予測してください。** ①は 2000000 になるでしょうか。②で同じ番号を2人が取ることはあるでしょうか。

In [ ]:
%%writefile ex02a.cpp
#include <iostream>
#include <thread>
#include <vector>

const int NMAX = 200000;

// ---- ① 共有カウンタを、守らずに増やす ----
long counter = 0;
void add_many() { for (int i = 0; i < 1000000; i++) counter++; }      // 守っていない

// ---- ② 共有の「次に取る番号」を、守らずに2人で進める ----
int idx = 0;                          // 次に取り出す番号（共有）
std::vector<int> got(NMAX, 0);        // 何番を何回取ったか
void worker() {
    while (true) {
        if (idx >= NMAX) return;      // ← まだ残っているか確かめて…
        int i = idx;                  // ← 自分の分として取って…
        idx++;                        // ← 番号を1つ進める
        got[i]++;                     //    この3行の「すきま」が危ない
    }
}

int main() {
    std::thread a1(add_many), a2(add_many);
    a1.join(); a2.join();
    std::cout << "① counter = " << counter << "   （期待値 2000000）\n" << std::flush;

    std::thread w1(worker), w2(worker);
    w1.join(); w2.join();
    int dup = 0, lost = 0;
    for (int v : got) { if (v > 1) dup++; if (v == 0) lost++; }
    std::cout << "② 2人が同じ番号を取ってしまった回数 = " << dup << "\n"
              << "   誰も取らなかった番号の数         = " << lost << "\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex02a.cpp -o ex02a
!for i in 1 2 3; do ./ex02a; echo ---; done

### 結果 ―― 毎回ちがう値になり、しかも足りない

```
① counter = 1054812   （期待値 2000000）
② 2人が同じ番号を取ってしまった回数 = 35519
   誰も取らなかった番号の数         = 14
```

**①の理由。** `counter++` は1行ですが、CPU から見ると **①読む ②足す ③書く** の3ステップです。

```
スレッドA: 読む(=100) 足す(=101)            書く(101)
スレッドB:            読む(=100) 足す(=101) 書く(101)
                                            ↑ 2回足したのに 101。1回分が消えた
```

**②の理由も同じ**です。「確かめる」と「使う」のあいだにすきまがあり、
そこに相手が割り込むと、2人が同じ番号を持ち帰ります。

> **「確かめてから使う」を分けてはいけない**（check-then-act）。
> 並行処理のバグの、かなりの割合がこれです。

やっかいなのは、**毎回起きるとは限らない**ことです。タイミング次第なので、
「たまに動く」「手元では再現しない」という最悪の形のバグになります。

**だから、段と段を生の `std::queue` でつなぐことはできません。**

## 2-2. 使うキュー ―― `ConcurrentQueue`

この演習で使うキューを渡します。**次のセルを実行すると `cq.h` が作られます。**
以降のプログラムは `#include "cq.h"` と書くだけで使えます。

**使うのは3つだけです。**

| 呼び方 | すること | 混んでいたら |
|---|---|---|
| `q.push(v)` | 1個入れる | **満杯なら空くまで待つ** |
| `T v = q.pop()` | 1個取り出す | **空なら来るまで待つ**（失敗しない） |
| `q.size()` | いま並んでいる個数 | 待たない。**観測用** |

そして、**知っておくべきことは3つだけ**です。

1. **待つのはキューの仕事。** 使う側は `push` と `pop` を呼ぶだけでよい
2. **容量（上限）がある。** `ConcurrentQueue<int> q(3);` の `3` が上限です。
   満杯なら入れる側が待たされます
3. **`size()` は「見る」ためだけのもの。** その値をもとに**判断して動いてはいけません**
   （2-1 の②と同じ罠になります ―― 発展課題2-1）

> **中身の仕組みを知りたい人は発展課題2-2へ。知らなくても使えます。**

In [ ]:
%%writefile cq.h
// ============================================================
//  cq.h ―― この演習で使うスレッドセーフなキュー
//  中身は読まなくてよい。使うのは push / pop / size の3つだけ。
// ============================================================
#pragma once
#include <queue>
#include <mutex>
#include <condition_variable>

template <typename T>
class ConcurrentQueue {
public:
    explicit ConcurrentQueue(std::size_t capacity) : capacity_(capacity) {}

    // 入れる。満杯なら空くまで待つ
    void push(const T& v) {
        std::unique_lock<std::mutex> lk(mtx_);
        can_push_.wait(lk, [this] { return q_.size() < capacity_; });
        q_.push(v);
        if (q_.size() > peak_) peak_ = q_.size();
        lk.unlock();
        can_pop_.notify_one();
    }

    // 取り出す。空なら来るまで待つ（失敗しない）
    T pop() {
        std::unique_lock<std::mutex> lk(mtx_);
        can_pop_.wait(lk, [this] { return !q_.empty(); });
        T v = q_.front(); q_.pop();
        lk.unlock();
        can_push_.notify_one();
        return v;
    }

    std::size_t size() const { std::lock_guard<std::mutex> g(mtx_); return q_.size(); }
    std::size_t peak() const { std::lock_guard<std::mutex> g(mtx_); return peak_; }   // 観測用：並んだ最大数

private:
    std::queue<T> q_;
    std::size_t capacity_;
    std::size_t peak_ = 0;
    mutable std::mutex mtx_;
    std::condition_variable can_pop_;    // 「取り出せるようになった」
    std::condition_variable can_push_;   // 「入れられるようになった」
};

## 2-3. 【予測クイズ】使ってみる

作る側が **10ms に1個**、使う側が **1個 50ms**（作る側のほうが5倍速い）。
容量 **3** のキューでつないで、30個流します。

**実行前に予測してください。**

- 30個を処理し終えるのに何 ms かかるでしょうか
- 途中でキューに並んでいる個数は、いくつくらいでしょうか

In [ ]:
%%writefile ex02b.cpp
#include <iostream>
#include <thread>
#include <chrono>
#include "cq.h"                                  // ← これだけで使える
using namespace std::chrono;

ConcurrentQueue<int> q(3);                       // 容量3のキュー

int main() {
    auto t0 = steady_clock::now();

    // 作る側：10ms に1個つくって push する
    std::thread producer([&] {
        for (int i = 1; i <= 30; i++) {
            std::this_thread::sleep_for(milliseconds(10));
            q.push(i);                           // 満杯なら、勝手に待ってくれる
        }
    });

    // 使う側：1個を 50ms かけて処理する（作る側より5倍おそい）
    std::thread consumer([&] {
        for (int i = 0; i < 30; i++) {
            int v = q.pop();                     // 空なら、勝手に待ってくれる
            std::this_thread::sleep_for(milliseconds(50));
            (void)v;
        }
    });

    // 観測係：0.2秒ごとに「いま何個並んでいるか」を見に行くだけ
    std::thread watcher([&] {
        for (int i = 0; i < 8; i++) {
            std::this_thread::sleep_for(milliseconds(200));
            std::cout << "  いまキューに並んでいる数 = " << q.size() << "\n";
        }
    });

    producer.join(); consumer.join(); watcher.join();

    std::cout << "\n30個を処理するのにかかった時間 = "
              << duration_cast<milliseconds>(steady_clock::now() - t0).count() << " ms\n";
    std::cout << "キューに並んだ最大の個数 = " << q.peak() << " 個（容量は 3）\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex02b.cpp -o ex02b && ./ex02b

### 結果 ―― 速さは遅い側で決まり、行列は容量で止まる

```
  いまキューに並んでいる数 = 3
  ...
30個を処理するのにかかった時間 = 1601 ms
キューに並んだ最大の個数 = 3 個（容量は 3）
```

**所要時間 約1500ms** = 30個 × 50ms。**遅いほうの段で決まっています。**
作る側が5倍速くても、全体は速くなりません（演習1のボトルネックの話そのものです）。

そして**先へ進んだ分は、速さにならずにキューに積み上がるだけ**です。
容量3を付けたので、そこで止まりました。もし上限がなければ、こうなります。

- 1フレーム 512×256×3バイト ≒ **384KB**
- 作る側が毎秒15フレーム速いなら、毎秒 5.8MB ずつ増える
- 1分で 340MB、10分で 3.4GB ⇒ **しばらく走らせると落ちる**

しかも落ちる前から壊れています。1000フレーム並んでいたら、
いま入れたフレームが画面に出るのは **33秒後**です。

> **容量を決めないということは、遅れとメモリの上限を決めないということ。**

満杯のとき入れる側が待たされる ―― この「遅い側の都合が速い側に伝わる」しくみを
**バックプレッシャ（backpressure）** と呼びます。
**では容量はいくつにすればよいのか。それが演習3です。**

## まとめ

- 守られていない共有データは**壊れる**。「確かめてから使う」を分けてはいけない
- 段と段は `ConcurrentQueue` でつなぐ。**使うのは `push` / `pop` / `size` の3つだけ**
- **待つのはキューの仕事。** 満杯なら `push` が、空なら `pop` が待ってくれる
- **容量**は、遅れとメモリの上限を決めるためにある

**次は演習3** ―― 容量をいくつにするか、段をどう分けるかを測って決めます。

> 🧩 発展課題は `adv02_queue.ipynb`（`size()` を判断に使ってはいけない理由、
> キューの中身、待つ `pop` と待たない `pop`）。